In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from pathlib import Path
import ctypes
import joblib
import numpy as np
import pandas as pd


# Ask Windows to render the Tk interface sharply on high-DPI displays.
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except (AttributeError, OSError):
    try:
        ctypes.windll.user32.SetProcessDPIAware()
    except (AttributeError, OSError):
        pass


# ============================================================
# MODEL FILE LOCATION
# ============================================================

MODEL_FILENAME = "SVR_final_pipeline.joblib"

# The order and spelling must exactly match the SVR training code.
FEATURE_NAMES = [
    "L",
    "Do",
    "L/Do",
    "to",
    "Di",
    "ti",
    "fc",
    "fyo",
    "fyi",
]

# Simple professional colour palette.
BACKGROUND_COLOR = "#EEF3F7"
PANEL_COLOR = "#FFFFFF"
PRIMARY_COLOR = "#173F5F"
ACCENT_COLOR = "#247BA0"
LIGHT_ACCENT_COLOR = "#DCEAF2"
SUCCESS_COLOR = "#238636"
RESULT_COLOR = "#C0392B"


def find_model_file():
    """
    Search for the saved model in common locations.

    Search order:
    1. Folder containing this Python script
    2. Current working directory (recommended for Jupyter Notebook)
    3. Desktop
    4. OneDrive Desktop
    5. Manual file selection
    """

    possible_paths = [
        Path.cwd() / MODEL_FILENAME,
        Path.home() / "Desktop" / MODEL_FILENAME,
        Path.home() / "OneDrive" / "Desktop" / MODEL_FILENAME,
    ]

    # When running as a normal .py file, also check the script folder.
    try:
        script_folder = Path(__file__).resolve().parent
        possible_paths.insert(0, script_folder / MODEL_FILENAME)
    except NameError:
        # __file__ is unavailable inside Jupyter Notebook.
        pass

    # Return the first valid path.
    for path in possible_paths:
        if path.is_file():
            return path

    # If the model was not found, ask the user to select it.
    selected_file = filedialog.askopenfilename(
        title="Select the SVR-CFDSST model file",
        filetypes=[
            ("Joblib model", "*.joblib"),
            ("All files", "*.*"),
        ],
    )

    if selected_file:
        return Path(selected_file)

    return None


def load_model():
    """Find and load the trained SVR-CFDSST model."""

    model_file = find_model_file()

    if model_file is None:
        messagebox.showerror(
            "Model File Not Found",
            f"The model file '{MODEL_FILENAME}' could not be found.\n\n"
            "Please place it in the same folder as this notebook or "
            "Python file, and then run the program again.",
        )
        return None, None

    try:
        loaded_model = joblib.load(model_file)
        return loaded_model, model_file

    except Exception as error:
        messagebox.showerror(
            "Model Loading Error",
            f"The model file was found but could not be loaded:\n\n"
            f"{model_file}\n\n"
            f"Error details:\n{error}",
        )
        return None, model_file


# ============================================================
# GUI FUNCTIONS
# ============================================================

def predict():
    """Read eight independent inputs and derive L/Do as the ninth feature."""

    if model is None:
        messagebox.showerror(
            "Model Error",
            "The prediction model has not been loaded.",
        )
        return

    try:
        # Read eight independent fields. The displayed X3 ratio is derived.
        input_values = [
            float(entry_x1.get().strip()),  # L
            float(entry_x2.get().strip()),  # Do
            float(entry_x3.get().strip()),  # to
            float(entry_x4.get().strip()),  # Di
            float(entry_x5.get().strip()),  # ti
            float(entry_x6.get().strip()),  # fc
            float(entry_x7.get().strip()),  # fyo
            float(entry_x8.get().strip()),  # fyi
        ]

        # All geometric and material inputs must be positive.
        if any(value <= 0 for value in input_values):
            raise ValueError("All input values must be greater than zero.")

        L, Do, to, Di, ti, fc, fyo, fyi = input_values
        update_ratio()

        # Basic CFDSST geometric-validity checks.
        if 2.0 * to >= Do:
            raise ValueError(
                "The outer-tube thickness is invalid: 2tₒ must be less than Dₒ."
            )

        if 2.0 * ti >= Di:
            raise ValueError(
                "The inner-tube thickness is invalid: 2tᵢ must be less than Dᵢ."
            )

        if Di >= Do - 2.0 * to:
            raise ValueError(
                "The inner tube does not fit inside the outer tube: "
                "Dᵢ must be less than Dₒ − 2tₒ."
            )

        # Feature order: X1 L, X2 Do, X3 L/Do, X4 to, X5 Di,
        # X6 ti, X7 fc, X8 fyo, X9 fyi.
        # Use a DataFrame because the trained ColumnTransformer selects inputs
        # by their exact column names.
        # L/Do is shown as X3 and calculated automatically from L and Do.
        model_values = [L, Do, L / Do, to, Di, ti, fc, fyo, fyi]
        model_input = pd.DataFrame([model_values], columns=FEATURE_NAMES)

        # Make the prediction.
        prediction_result = model.predict(model_input)

        # MultiOutputRegressor returns a two-dimensional one-target output;
        # reshape safely to obtain one numerical prediction.
        prediction = float(
            np.asarray(prediction_result, dtype=float).reshape(-1)[0]
        )

        # Display the predicted ultimate axial capacity.
        output_label.config(text=f"{prediction:,.4f}")

    except ValueError as error:
        messagebox.showerror(
            "Input Error",
            "Please enter a valid numerical value for every input parameter.\n\n"
            f"Details:\n{error}",
        )

    except Exception as error:
        messagebox.showerror(
            "Prediction Error",
            f"The prediction could not be completed.\n\n"
            f"Error details:\n{error}",
        )


def clear_inputs():
    """Clear all input fields and the prediction result."""

    for entry in input_entries:
        if entry is entry_ratio:
            entry.configure(state="normal")
        entry.delete(0, tk.END)
        if entry is entry_ratio:
            entry.configure(state="readonly")

    output_label.config(text="")
    entry_x1.focus_set()


# ============================================================
# CREATE THE MAIN WINDOW
# ============================================================

root = tk.Tk()
root.title("Circular CFDSST Capacity Prediction")
root.geometry("1660x920")
root.minsize(1500, 850)
root.configure(bg=BACKGROUND_COLOR)

# Keep the interface simple while applying one consistent colour theme.
style = ttk.Style(root)
try:
    style.theme_use("clam")
except tk.TclError:
    pass

style.configure(
    "TFrame",
    background=BACKGROUND_COLOR,
)
style.configure(
    "Panel.TLabelframe",
    background=PANEL_COLOR,
    bordercolor="#B8C6D1",
    relief="solid",
)
style.configure(
    "Panel.TLabelframe.Label",
    background=PANEL_COLOR,
    foreground=PRIMARY_COLOR,
    font=("Consolas", 11, "bold"),
)
style.configure(
    "Panel.TLabel",
    background=PANEL_COLOR,
    foreground="#17212B",
)
style.configure(
    "TEntry",
    fieldbackground="#FFFFFF",
    foreground="#17212B",
    bordercolor="#9FB2C1",
    lightcolor=ACCENT_COLOR,
    darkcolor="#9FB2C1",
)
style.configure(
    "Primary.TButton",
    font=("Consolas", 11, "bold"),
    foreground="white",
    background=ACCENT_COLOR,
    bordercolor=ACCENT_COLOR,
    padding=(18, 7),
)
style.map(
    "Primary.TButton",
    background=[("active", PRIMARY_COLOR)],
)
style.configure(
    "Simple.TButton",
    font=("Consolas", 11),
    foreground=PRIMARY_COLOR,
    background="#FFFFFF",
    bordercolor="#9FB2C1",
    padding=(18, 7),
)
style.map(
    "Simple.TButton",
    background=[("active", LIGHT_ACCENT_COLOR)],
)


# ============================================================
# LOAD THE MODEL
# ============================================================

model, model_path = load_model()


# ============================================================
# TITLE
# ============================================================

title_label = tk.Label(
    root,
    text="SVR Prediction of Circular CFDSST Columns",
    font=("Consolas", 18, "bold"),
    foreground="white",
    background=PRIMARY_COLOR,
    anchor="center",
    pady=12,
)

title_label.pack(
    fill="x",
    padx=0,
    pady=(0, 8),
)


# ============================================================
# MODEL STATUS
# ============================================================

if model is not None:
    model_status_text = f"Model loaded successfully: {model_path}"
    model_status_color = "green"
else:
    model_status_text = "Model not loaded"
    model_status_color = "red"

model_status_label = tk.Label(
    root,
    text=model_status_text,
    font=("Consolas", 10),
    foreground=model_status_color,
    background=BACKGROUND_COLOR,
    anchor="w",
)

model_status_label.pack(
    fill="x",
    padx=15,
    pady=(0, 5),
)


# ============================================================
# INPUT PARAMETERS
# ============================================================

input_frame = ttk.LabelFrame(
    root,
    text="Input Parameters",
    padding=(10, 10, 10, 10),
    style="Panel.TLabelframe",
)

input_frame.pack(
    padx=15,
    pady=10,
    fill="x",
)

input_frame.columnconfigure(0, weight=1)
input_frame.columnconfigure(1, weight=0)
input_frame.columnconfigure(2, weight=0)


# Nine displayed features; L/Do is calculated automatically to prevent
# inconsistent values and to match the nine-feature training pipeline.
PARAMETERS = [
    ("Column length (L, mm)", "L"),
    ("Diameter of outer steel tube (Dₒ, mm)", "Do"),
    ("Slenderness ratio (L/Dₒ)", "L/Do"),
    ("Thickness of outer steel tube (tₒ, mm)", "to"),
    ("Diameter of inner steel tube (Dᵢ, mm)", "Di"),
    ("Thickness of inner steel tube (tᵢ, mm)", "ti"),
    ("Concrete compressive strength (f꜀, MPa)", "fc"),
    ("Yield strength of outer steel tube (fᵧₒ, MPa)", "fyo"),
    ("Yield strength of inner steel tube (fᵧᵢ, MPa)", "fyi"),
]

input_entries = []
for idx, (description, feature) in enumerate(PARAMETERS):
    ttk.Label(
        input_frame, text=f"X{idx+1}: {description}",
        font=("Times New Roman", 14), style="Panel.TLabel",
    ).grid(row=idx, column=0, padx=5, pady=5, sticky=tk.W)
    entry = ttk.Entry(input_frame, font=("Times New Roman", 14), width=20)
    entry.grid(row=idx, column=1, padx=5, pady=5)
    input_entries.append(entry)

(entry_x1, entry_ratio, entry_x2, entry_x3, entry_x4,
 entry_x5, entry_x6, entry_x7, entry_x8) = (
    input_entries[0], input_entries[2], input_entries[1],
    input_entries[3], input_entries[4], input_entries[5],
    input_entries[6], input_entries[7], input_entries[8]
)

# X3 is a read-only computed feature; no manual entry is required.
entry_ratio.configure(state="readonly")

def update_ratio(*_args):
    try:
        length = float(entry_x1.get().strip())
        diameter = float(entry_x2.get().strip())
        value = f"{length / diameter:.1f}" if diameter > 0 else ""
    except (ValueError, ZeroDivisionError):
        value = ""
    entry_ratio.configure(state="normal")
    entry_ratio.delete(0, tk.END)
    entry_ratio.insert(0, value)
    entry_ratio.configure(state="readonly")

entry_x1.bind("<KeyRelease>", update_ratio)
entry_x2.bind("<KeyRelease>", update_ratio)


# ============================================================
# ENGINEERING SCHEMATIC - SPACED TO PREVENT OVERLAPS
# ============================================================
# All drawing coordinates are within a 650 x 485 canvas.
# Separate elevation, cross-section and legend zones.
schematic = tk.Canvas(
    input_frame, width=650, height=485, background=PANEL_COLOR,
    highlightthickness=1, highlightbackground=LIGHT_ACCENT_COLOR,
)
schematic.grid(row=0, column=2, rowspan=9, padx=(25, 8), pady=0,
               sticky="nsew")

FNT = "Times New Roman"

def drawing_text(x, y, value, size=13, color=PRIMARY_COLOR,
                 anchor="center", weight="normal", italic=False):
    font = (FNT, size, ("bold italic" if weight == "bold" and italic else
                         "bold" if weight == "bold" else
                         "italic" if italic else "normal"))
    return schematic.create_text(x, y, text=value, font=font,
                                 fill=color, anchor=anchor)

# Header and two panel headings (dedicated rows).
drawing_text(325, 19, "Circular CFDSST geometry and axial loading",
             14, weight="bold")
drawing_text(138, 54, "(a) Axially loaded column", 12, italic=True)
drawing_text(458, 54, "(b) Cross-section", 12, italic=True)

# Left zone: column elevation, x=25..260, y=90..375.
cl, cr, ct, cb = 112, 166, 137, 337
schematic.create_rectangle(102, 126, 176, 137, fill="#667681",
                           outline=PRIMARY_COLOR, width=2)
schematic.create_rectangle(102, 337, 176, 348, fill="#667681",
                           outline=PRIMARY_COLOR, width=2)
schematic.create_rectangle(cl, ct, cr, cb, fill="#D9E1E6",
                           outline=PRIMARY_COLOR, width=4)
schematic.create_line(cl + 9, ct, cl + 9, cb, fill=ACCENT_COLOR, width=3)
schematic.create_line(cr - 9, ct, cr - 9, cb, fill=ACCENT_COLOR, width=3)
schematic.create_oval(cl, ct - 5, cr, ct + 5, fill="#D9E1E6",
                     outline=PRIMARY_COLOR, width=2)
schematic.create_oval(cl, cb - 5, cr, cb + 5, fill="#D9E1E6",
                     outline=PRIMARY_COLOR, width=2)
# Compression arrows occupy clear space above and below the plates.
schematic.create_line(139, 83, 139, 118, arrow=tk.LAST,
                      arrowshape=(12, 14, 5), fill=RESULT_COLOR, width=3)
drawing_text(157, 98, "P", 15, RESULT_COLOR, italic=True)
schematic.create_line(139, 392, 139, 355, arrow=tk.LAST,
                      arrowshape=(12, 14, 5), fill=RESULT_COLOR, width=3)
drawing_text(157, 376, "P", 15, RESULT_COLOR, italic=True)
# L dimension is entirely left of the column.
schematic.create_line(68, ct, 68, cb, arrow=tk.BOTH,
                      arrowshape=(9, 11, 4), fill=RESULT_COLOR, width=2)
schematic.create_line(68, ct, 105, ct, fill=RESULT_COLOR)
schematic.create_line(68, cb, 105, cb, fill=RESULT_COLOR)
drawing_text(51, (ct + cb) / 2, "L", 17, RESULT_COLOR, italic=True)

# Right zone: cross-section, x=310..620, y=90..380.
cx, cy = 452, 237
ro, rc, ri, ric = 82, 68, 39, 27
schematic.create_oval(cx-ro, cy-ro, cx+ro, cy+ro,
                     fill=PRIMARY_COLOR, outline=PRIMARY_COLOR)
schematic.create_oval(cx-rc, cy-rc, cx+rc, cy+rc,
                     fill="#D9E1E6", outline=PRIMARY_COLOR)
schematic.create_oval(cx-ri, cy-ri, cx+ri, cy+ri,
                     fill=ACCENT_COLOR, outline=ACCENT_COLOR)
schematic.create_oval(cx-ric, cy-ric, cx+ric, cy+ric,
                     fill="white", outline=ACCENT_COLOR)
schematic.create_line(cx-ro-8, cy, cx+ro+8, cy,
                      fill="#8798A6", dash=(3, 3))
schematic.create_line(cx, cy-ro-8, cx, cy+ro+8,
                      fill="#8798A6", dash=(3, 3))
# Outer diameter dimension: dedicated gap above the section.
dy = 118
schematic.create_line(cx-ro, dy, cx+ro, dy, arrow=tk.BOTH,
                      arrowshape=(8, 10, 4), fill=RESULT_COLOR, width=2)
schematic.create_line(cx-ro, dy, cx-ro, cy-ro-4, fill=RESULT_COLOR)
schematic.create_line(cx+ro, dy, cx+ro, cy-ro-4, fill=RESULT_COLOR)
drawing_text(cx, 98, "Dₒ", 16, RESULT_COLOR, italic=True)
# Inner diameter dimension: dedicated gap below the section.
dy2 = 353
schematic.create_line(cx-ri, dy2, cx+ri, dy2, arrow=tk.BOTH,
                      arrowshape=(8, 10, 4), fill=RESULT_COLOR, width=2)
schematic.create_line(cx-ri, cy+ri+2, cx-ri, dy2, fill=RESULT_COLOR)
schematic.create_line(cx+ri, cy+ri+2, cx+ri, dy2, fill=RESULT_COLOR)
drawing_text(cx, 372, "Dᵢ", 16, RESULT_COLOR, italic=True)
# Thickness callouts are on the right with independent leader positions.
schematic.create_line(cx+60, cy-55, 563, 167, arrow=tk.FIRST,
                      arrowshape=(9, 11, 4), fill="#C47A00", width=2)
drawing_text(572, 160, "tₒ", 15, "#A56400", anchor="w", italic=True)
schematic.create_line(cx+30, cy-24, 563, 220, arrow=tk.FIRST,
                      arrowshape=(9, 11, 4), fill="#C47A00", width=2)
drawing_text(572, 218, "tᵢ", 15, "#A56400", anchor="w", italic=True)

# Legend: three separate rows below the drawings, no diagram overlap.
schematic.create_line(20, 402, 630, 402, fill="#D5E0E9")
legend_items = [
    (421, PRIMARY_COLOR, "Outer steel: fᵧₒ"),
    (443, ACCENT_COLOR, "Inner steel: fᵧᵢ"),
    (465, "#D9E1E6", "Concrete: f꜀"),
]
for y, color, label in legend_items:
    schematic.create_rectangle(244, y-7, 258, y+7, fill=color,
                               outline=PRIMARY_COLOR)
    drawing_text(269, y, label, 12, "#17212B", anchor="w", italic=True)

# ============================================================
# PREDICTION RESULT
# ============================================================

output_frame = ttk.LabelFrame(
    root,
    text="Prediction Result",
    padding=(10, 10, 10, 10),
    style="Panel.TLabelframe",
)

output_frame.pack(
    padx=15,
    pady=10,
    fill="x",
)

ttk.Label(
    output_frame,
    text="Predicted Ultimate Axial Capacity (Nᵤ, MN) =",
    font=("Times New Roman", 14),
    style="Panel.TLabel",
).grid(row=0, column=0, padx=5, pady=5, sticky=tk.W)

output_label = ttk.Label(
    output_frame,
    text="",
    font=("Times New Roman", 14, "bold"),
    foreground=RESULT_COLOR,
    background=PANEL_COLOR,
)

output_label.grid(
    row=0,
    column=1,
    padx=5,
    pady=5,
    sticky=tk.W,
)


# ============================================================
# BUTTONS
# ============================================================

buttons_frame = ttk.Frame(root)
buttons_frame.pack(pady=10)

predict_button = ttk.Button(
    buttons_frame,
    text="Predict",
    command=predict,
    style="Primary.TButton",
)
predict_button.grid(row=0, column=0, padx=8)

clear_button = ttk.Button(
    buttons_frame,
    text="Clear",
    command=clear_inputs,
    style="Simple.TButton",
)
clear_button.grid(row=0, column=1, padx=8)

exit_button = ttk.Button(
    buttons_frame,
    text="Exit",
    command=root.destroy,
    style="Simple.TButton",
)
exit_button.grid(row=0, column=2, padx=8)


# Allow prediction by pressing Enter.
root.bind("<Return>", lambda event: predict())

# Place the cursor in the first input field.
entry_x1.focus_set()

# Start the GUI event loop.
root.mainloop()